# Fine-tune Cross-Encoder v0.6 - MSE + Spearman, 15 Epochs

Extended run to test if validation and test metrics continue improving beyond epoch 10.

| | |
|---|---|
| **Dataset** | v0.5 — 9,350 train / 2,000 validation / 2,000 test (13,350 pairs, balanced 20% per class) |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `MSELoss` |
| **Evaluator** | Spearman correlation |
| **Epochs** | **15** (extended from 10) |
| **max_length** | 512 tokens |
| **Branch** | `experiment/cross-encoder-v0.6` |

**Goal**: Check if continuing training beyond epoch 10 yields better test LabelAcc.

In [1]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.6
!git pull origin experiment/cross-encoder-v0.6

Cloning into '/content/Ai-Recruiter-Mini-Ai-Service'...
remote: Enumerating objects: 7356, done.
remote: Counting objects: 100% (337/337), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 7356 (delta 190), reused 171 (delta 153), pack-reused 7019 (from 2)
Receiving objects: 100% (7356/7356), 32.19 MiB | 12.25 MiB/s, done.
Resolving deltas: 100% (4366/4366), done.
/content/Ai-Recruiter-Mini-Ai-Service
Branch 'experiment/cross-encoder-v0.6' set up to track remote branch 'experiment/cross-encoder-v0.6' from 'origin'.
Switched to a new branch 'experiment/cross-encoder-v0.6'
From https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service
 * branch            experiment/cross-encoder-v0.6 -> FETCH_HEAD
Already up to date.


In [2]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.7/517.7 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.2/375.2 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.5/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")

train       :  9350 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
validation  :  2000 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']
test        :  2000 pairs  | keys: ['pair_id', 'cv_text', 'jd_text', 'score', 'label', 'true_label']


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


## Helper Functions

In [5]:
import json
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample
from typing import Any

# Evaluator: CECorrelationEvaluator (Spearman)
class CECorrelationEvaluator:
    """Evaluate cross-encoder using Spearman correlation."""
    def __init__(self, sentence_pairs: list, labels_0_1: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1 = labels_0_1
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CECorrelationEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_1=[ex.label for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0

def load_jsonl(path: str) -> list[dict[str, Any]]:
    """Load JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def load_dataset(data_dir: str) -> tuple[list[InputExample], list[InputExample], list[InputExample]]:
    """Load train/val/test InputExample lists."""
    train_data = load_jsonl(f"{data_dir}/cross_encoder_train.jsonl")
    val_data = load_jsonl(f"{data_dir}/cross_encoder_validation.jsonl")
    test_data = load_jsonl(f"{data_dir}/cross_encoder_test.jsonl")

    def to_input_examples(records):
        return [
            InputExample(texts=[rec['cv_text'], rec['jd_text']], label=rec['label'])
            for rec in records
        ]

    return to_input_examples(train_data), to_input_examples(val_data), to_input_examples(test_data)

def compute_metrics(model: CrossEncoder, examples: list[InputExample], batch_size: int = 32) -> dict:
    """Compute MAE, RMSE, LabelAcc on examples."""
    preds = model.predict([ex.texts for ex in examples], batch_size=batch_size, show_progress_bar=False)
    preds_100 = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])

    mae = np.mean(np.abs(preds_100 - labels_100))
    rmse = np.sqrt(np.mean((preds_100 - labels_100) ** 2))
    label_acc = np.mean(np.abs(preds_100 - labels_100) <= 10)

    return {'MAE': mae, 'RMSE': rmse, 'LabelAcc': label_acc}

print("✅ Helper functions loaded.")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ Helper functions loaded.


## Load Dataset

In [6]:
train_examples, val_examples, test_examples = load_dataset('datasets/versions/v0.5/cross_encoder')

print(f"Train: {len(train_examples)} pairs")
print(f"Val:   {len(val_examples)} pairs")
print(f"Test:  {len(test_examples)} pairs")
print(f"Total: {len(train_examples) + len(val_examples) + len(test_examples)} pairs")

Train: 9350 pairs
Val:   2000 pairs
Test:  2000 pairs
Total: 13350 pairs


## Run 1: MSELoss + Spearman, 15 Epochs

In [7]:
import torch
from torch.utils.data import DataLoader

run_name = "v0.6-mse-spearman-15ep"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-12-v2',
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

# ✅ USE SIMPLE EVALUATOR (like v0.6-10ep)
evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# Calculate warmup steps (10% of total steps)
total_steps = (len(train_examples) // 16 + 1) * 15
warmup_steps = int(total_steps * 0.1)

print(f"🚀 Starting {run_name}...")
model.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=15,
    loss_fct=torch.nn.MSELoss(),
    warmup_steps=warmup_steps,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ {run_name} completed.")
print(f"\nValidation metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")
print(f"\nTest metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

results = {
    'run': run_name,
    'loss': 'MSE',
    'evaluator': 'Spearman',
    'epochs': 15,
    'val': val_metrics,
    'test': test_metrics
}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]

Iteration:   0%|          | 0/585 [00:00<?, ?it/s]


✅ v0.6-mse-spearman-15ep completed.

Validation metrics:
  MAE: 9.6157
  RMSE: 13.0259
  LabelAcc: 0.6380

Test metrics:
  MAE: 9.9887
  RMSE: 13.5072
  LabelAcc: 0.6160


## Save Report

In [8]:
import os
import json
from pathlib import Path

# Create reports directory
os.makedirs('artifacts/reports', exist_ok=True)

# Save report
report = {
    'base_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
    'dataset_version': 'v0.5',
    'dataset_size': {
        'train': len(train_examples),
        'val': len(val_examples),
        'test': len(test_examples),
        'total': len(train_examples) + len(val_examples) + len(test_examples)
    },
    'run': results['run'],
    'loss': results['loss'],
    'evaluator': results['evaluator'],
    'epochs': results['epochs'],
    'metrics': {
        'validation': {k: float(v) for k, v in results['val'].items()},
        'test': {k: float(v) for k, v in results['test'].items()}
    },
    'model_path': 'artifacts/models/cross-encoder-cv-jd-v0.6-mse-spearman-15ep'
}

report_path = 'artifacts/reports/fine_tune_cross_encoder_v0.6_mse_spearman_15ep_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"✅ Report saved to {report_path}")
print(f"\n📊 Summary:")
print(f"  Base model: cross-encoder/ms-marco-MiniLM-L-12-v2")
print(f"  Loss: {results['loss']}")
print(f"  Evaluator: {results['evaluator']}")
print(f"  Epochs: {results['epochs']}")
print(f"  Final Test LabelAcc: {results['test']['LabelAcc']:.4f} ({results['test']['LabelAcc']*100:.2f}%)")

✅ Report saved to artifacts/reports/fine_tune_cross_encoder_v0.6_mse_spearman_15ep_report.json

📊 Summary:
  Base model: cross-encoder/ms-marco-MiniLM-L-12-v2
  Loss: MSE
  Evaluator: Spearman
  Epochs: 15
  Final Test LabelAcc: 0.6160 (61.60%)


## Save to Google Drive (Optional)

In [10]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"
os.makedirs(f"{drive_base}/models", exist_ok=True)
os.makedirs(f"{drive_base}/reports", exist_ok=True)

# Save model
src_dir = 'artifacts/models/cross-encoder-cv-jd-v0.6-mse-spearman-15ep'
dest_dir = f"{drive_base}/models/cross-encoder-cv-jd-v0.6-mse-spearman-15ep"
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
shutil.copytree(src_dir, dest_dir)
print(f"✅ Model saved to Google Drive")

# Copy report
src = 'artifacts/reports/fine_tune_cross_encoder_v0.6_mse_spearman_15ep_report.json'
dest = f"{drive_base}/reports/fine_tune_cross_encoder_v0.6_mse_spearman_15ep_report.json"
shutil.copy(src, dest)
print(f"✅ Report saved to Google Drive")

print(f"\n✅ Done! Model and report saved to Google Drive!")

Mounted at /content/drive
✅ Model saved to Google Drive
✅ Report saved to Google Drive

✅ Done! Model and report saved to Google Drive!
